In [ ]:
# Dados/Web site:
# https://dadosabertos.saude.gov.br/dataset/arboviroses-dengue

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import gc

print(f"Versão:")
print(f'Python: {sys.version}')
print(f'Pandas: {pd.__version__}')
print(f'Numpy: {np.__version__}')

Versão:
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Pandas: 2.2.2
Numpy: 2.0.2


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
path = '/content/drive/MyDrive/ML-UFF'

Mounted at /content/drive


In [ ]:
# Filtro do código por estado:
# SG_UF
# Códigos de UF (IBGE) mais comuns:
# 33: Rio de Janeiro
# 35: São Paulo
# 31: Minas Gerais
# 41: Paraná

In [ ]:
# O código do município no SINAN geralmente é salvo na coluna 'ID_MUNIC_RES' ou 'ID_MN_RESE'
# Filtrando para a cidade do Rio de Janeiro

# df_rio = df_local[df_local['ID_MN_RESE'] == '330455']
# Rio de Janeiro (Capital): 330455
# Niterói: 330330

In [ ]:
def save_data_frame(df: pd.DataFrame, path_dataset: str):
  df.to_csv(path_dataset, index=True)

In [ ]:
def get_data_frame(path_dataset: str):
  data = pd.read_csv(path_dataset)
  return data

In [ ]:
def get_data_frame_colab(path: str, dataset: str) -> pd.DataFrame:
    path_name = f'{path}/{dataset}.csv'
    local_path = f'/content/{dataset}.csv'

    print(f"Copiando {dataset} para o disco local...")
    # Use o '$' para passar as variáveis do Python para o comando do sistema (Shell)
    !cp $path_name $local_path

    print(f"Carregando {dataset} na RAM...")
    df = pd.read_csv(local_path, low_memory=False)

    # 3. Deleta o arquivo temporário do disco para não lotar o HD do Colab
    if os.path.exists(local_path):
        os.remove(local_path)
        print(f"Arquivo local {dataset}.csv removido para liberar espaço.")
    return df

In [ ]:
def preparar_dados_sinan(df, codigo_municipio=330455, init_date = '2020-01-01') -> pd.DataFrame:
  """
  Filtra e prepara o dataframe do SINAN tratando erros de valores nulos.
  """
  # Criamos uma cópia para não alterar o dataframe original
  df_copy = df.copy()

  # 1. Tratamento do ID_MN_RESI (Garante que não dê erro no filtro)
  # Convertemos para string e removemos o '.0' que o pandas as vezes coloca em floats
  df_copy['ID_MN_RESI'] = df_copy['ID_MN_RESI'].astype(str).str.replace('.0', '', regex=False)

  # Filtrar pelo código (como string para evitar o IntCastingNaNError)
  codigo_str = str(int(codigo_municipio))
  df_filtrado = df_copy[df_copy['ID_MN_RESI'] == codigo_str].copy()

  if df_filtrado.empty:
      print(f"Aviso: Nenhum dado encontrado para o município {codigo_municipio}")
      return df_filtrado

  # 2. Converter DT_SIN_PRI para datetime
  # Se a data estiver em formatos estranhos, o errors='coerce' transforma em NaT (nulo)
  df_filtrado['DT_SIN_PRI'] = pd.to_datetime(df_filtrado['DT_SIN_PRI'], errors='coerce')

  # Remove registros onde a data de sintomas é nula
  df_filtrado = df_filtrado.dropna(subset=['DT_SIN_PRI'])


  # 3. Colunas importantes para ML
  colunas_ml = [
      'DT_SIN_PRI', 'NU_IDADE_N', 'CS_SEXO', 'CS_GESTANT',
      'FEBRE', 'MIALGIA', 'CEFALEIA', 'EXANTEMA', 'VOMITO', 'NAUSEA',
      'DOR_RETRO', 'ARTRALGIA', 'ARTRITE', 'CONJUNTVIT', 'PETEQUIA_N', 'LACO',
      'DIABETES', 'HIPERTENSA', 'RENAL', 'HEMATOLOG', 'HEPATOPAT',
      'CLASSI_FIN', 'EVOLUCAO'
  ]

  colunas_presentes = [c for c in colunas_ml if c in df_filtrado.columns]
  df_final = df_filtrado[colunas_presentes].copy()

  # 4. Ordenar e definir índice temporal
  df_final = df_final.sort_values('DT_SIN_PRI')
  df_final.set_index('DT_SIN_PRI', inplace=True)

  df_final = df_final[df_final.index >= init_date]

  return df_final

In [ ]:
rio_code = 330455

In [ ]:
dataset_names = ['DENGBR20','DENGBR21','DENGBR22','DENGBR23','DENGBR24','DENGBR25','DENGBR26']

In [ ]:
name_list = list()
for name in dataset_names:
  name_ = f'dengue_rj_{name[-2:]}.csv'
  path_dengue_rj = f'{path}/{name_}'
  print(f"Processando: {name}")
  temp_df = get_data_frame_colab(path, name)
  df = preparar_dados_sinan(temp_df, rio_code)
  del temp_df
  save_data_frame(df, path_dengue_rj)
  name_list.append(name_)


Processando: DENGBR25
Copiando DENGBR25 para o disco local...
Carregando DENGBR25 na RAM...
Arquivo local DENGBR25.csv removido para liberar espaço.
Processando: DENGBR26
Copiando DENGBR26 para o disco local...
Carregando DENGBR26 na RAM...
Arquivo local DENGBR26.csv removido para liberar espaço.


In [ ]:
name_list = list()
for name in dataset_names:
  name_ = f'dengue_rj_{name[-2:]}.csv'
  name_list.append(name_)

In [ ]:
name_list

['dengue_rj_20.csv',
 'dengue_rj_21.csv',
 'dengue_rj_22.csv',
 'dengue_rj_23.csv',
 'dengue_rj_24.csv',
 'dengue_rj_25.csv',
 'dengue_rj_26.csv']

In [ ]:
df_final = pd.DataFrame()
for name in name_list:
  path_dengue_rj = f'{path}/{name}'
  print(f"Buscando: {path_dengue_rj}")
  temp_df = get_data_frame(path_dengue_rj)
  df_final = pd.concat([df_final, temp_df], axis=0, ignore_index=True)
  del temp_df

Buscando: /content/drive/MyDrive/ML-UFF/dengue_rj_20.csv
Buscando: /content/drive/MyDrive/ML-UFF/dengue_rj_21.csv
Buscando: /content/drive/MyDrive/ML-UFF/dengue_rj_22.csv
Buscando: /content/drive/MyDrive/ML-UFF/dengue_rj_23.csv
Buscando: /content/drive/MyDrive/ML-UFF/dengue_rj_24.csv
Buscando: /content/drive/MyDrive/ML-UFF/dengue_rj_25.csv
Buscando: /content/drive/MyDrive/ML-UFF/dengue_rj_26.csv


In [ ]:
df_final.shape, df_final.columns

((150728, 23),
 Index(['DT_SIN_PRI', 'NU_IDADE_N', 'CS_SEXO', 'CS_GESTANT', 'FEBRE', 'MIALGIA',
        'CEFALEIA', 'EXANTEMA', 'VOMITO', 'NAUSEA', 'DOR_RETRO', 'ARTRALGIA',
        'ARTRITE', 'CONJUNTVIT', 'PETEQUIA_N', 'LACO', 'DIABETES', 'HIPERTENSA',
        'RENAL', 'HEMATOLOG', 'HEPATOPAT', 'CLASSI_FIN', 'EVOLUCAO'],
       dtype='object'))

In [ ]:
df_final.head()

,DT_SIN_PRI,NU_IDADE_N,CS_SEXO,CS_GESTANT,FEBRE,MIALGIA,CEFALEIA,EXANTEMA,VOMITO,NAUSEA,...,CONJUNTVIT,PETEQUIA_N,LACO,DIABETES,HIPERTENSA,RENAL,HEMATOLOG,HEPATOPAT,CLASSI_FIN,EVOLUCAO
0,2020-01-01,4062,F,5.0,1.0,1.0,1.0,1.0,2.0,2.0,...,2.0,2.0,2.0,1.0,2.0,2.0,2.0,2.0,5.0,1.0
1,2020-01-01,4062,F,5.0,1.0,1.0,1.0,2.0,1.0,1.0,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,5.0,1.0
2,2020-01-01,4039,M,6.0,1.0,1.0,1.0,2.0,2.0,2.0,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,10.0,1.0
3,2020-01-01,4030,F,5.0,1.0,1.0,2.0,2.0,2.0,2.0,...,2.0,2.0,2.0,2.0,1.0,2.0,2.0,2.0,10.0,9.0
4,2020-01-01,4024,M,6.0,1.0,1.0,1.0,2.0,2.0,2.0,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,10.0,NaN


In [ ]:
df_final.index.unique()

RangeIndex(start=0, stop=150728, step=1)

In [ ]:
# proporção (%) de dados faltantes
(df_final.isna().sum() / len(df_final)) * 100

,0
DT_SIN_PRI,0.000000
NU_IDADE_N,0.000000
CS_SEXO,0.000663
CS_GESTANT,0.016586
FEBRE,0.490287
MIALGIA,0.490287
CEFALEIA,0.490287
EXANTEMA,0.490287
VOMITO,0.490287
NAUSEA,0.490287


In [ ]:
df_final.isna().sum()

,0
DT_SIN_PRI,0
NU_IDADE_N,0
CS_SEXO,1
CS_GESTANT,25
FEBRE,739
MIALGIA,739
CEFALEIA,739
EXANTEMA,739
VOMITO,739
NAUSEA,739


In [ ]:
df_final.dropna(inplace=True)

In [ ]:
df_final.shape, df_final.columns

((132407, 23),
 Index(['DT_SIN_PRI', 'NU_IDADE_N', 'CS_SEXO', 'CS_GESTANT', 'FEBRE', 'MIALGIA',
        'CEFALEIA', 'EXANTEMA', 'VOMITO', 'NAUSEA', 'DOR_RETRO', 'ARTRALGIA',
        'ARTRITE', 'CONJUNTVIT', 'PETEQUIA_N', 'LACO', 'DIABETES', 'HIPERTENSA',
        'RENAL', 'HEMATOLOG', 'HEPATOPAT', 'CLASSI_FIN', 'EVOLUCAO'],
       dtype='object'))

In [ ]:
df_final.head()

,DT_SIN_PRI,NU_IDADE_N,CS_SEXO,CS_GESTANT,FEBRE,MIALGIA,CEFALEIA,EXANTEMA,VOMITO,NAUSEA,...,CONJUNTVIT,PETEQUIA_N,LACO,DIABETES,HIPERTENSA,RENAL,HEMATOLOG,HEPATOPAT,CLASSI_FIN,EVOLUCAO
0,2020-01-01,4062,F,5.0,1.0,1.0,1.0,1.0,2.0,2.0,...,2.0,2.0,2.0,1.0,2.0,2.0,2.0,2.0,5.0,1.0
1,2020-01-01,4062,F,5.0,1.0,1.0,1.0,2.0,1.0,1.0,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,5.0,1.0
2,2020-01-01,4039,M,6.0,1.0,1.0,1.0,2.0,2.0,2.0,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,10.0,1.0
3,2020-01-01,4030,F,5.0,1.0,1.0,2.0,2.0,2.0,2.0,...,2.0,2.0,2.0,2.0,1.0,2.0,2.0,2.0,10.0,9.0
6,2020-01-01,4032,M,6.0,1.0,1.0,2.0,1.0,2.0,2.0,...,2.0,2.0,2.0,2.0,1.0,2.0,2.0,2.0,10.0,1.0


In [ ]:
path_dengue_rj = '/content/drive/MyDrive/ML-UFF/dengue_rj_2020_2026.csv'

In [ ]:
save_data_frame(df_final, path_dengue_rj)